# Convolutional Neural Network to classify image classes: 
## Training set: CIFAR10
## Stack: PyTorch

- This notebooks documents the functions of the libraries used and codes as in line comment for context. 
- CPU was used for training. 
- There is also a cell at the last section where the code for reusing the model is also noted. Follow the instructions to reproduce the model and its uses. 

### Imports and Device Declaration

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim  # various optimization algorithms to update model weights during backpropagation
from torch.utils.data import random_split, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision
import torchvision.transforms as transforms  # preporcessing and data augmentation
from torchvision.datasets import ImageFolder
# import matplotlib.pyplot as plt
# import numpy as np
from pathlib import Path

# Configurations #
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 30
DEVICE = torch.device("cpu")
CHECKPOINT_DIR = Path("../Checkpoints")
CHECKPOINT_DIR.mkdir(parents = True, exist_ok = True)
# Exact precalculated values for CIFAR10
MEAN = (0.4914, 0.4822, 0.4465)  # Statistical mean
STD = (0.2023, 0.1994, 0.2010)   # Standard deviation
# NN learn best when input features are centered around 0. Applying normalization prevents features 
# with large magnitudes from dominating the gradient updates, which leads to much faster and more stable 
# convergence during training.

### Data Transformation
- The data was downloaded manually as the PyTorch mirror couldn't access it. The server was down at the time of building this model because of power issues in the canadian university hosting the dataset. 

In [19]:
# Transformation #
transform_train = transforms.Compose(   # chain conversion operations definer 
    [
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding = 4),
        transforms.RandomRotation(15), # applies in range [-15, 15]
        transforms.ToTensor(),         # PIL images or NumPy arrays into FloatTensor
        transforms.Normalize(MEAN, STD) # Standardize the data distribution : pixel values [0, 1]
    ]
)
transform_val_test = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD)
    ]
)

train_set = ImageFolder(root = '../Data/cifar10/train', transform = None)
test_set = ImageFolder(root = '../Data/cifar10/test', transform = None)

# Wrapper for Subset Transforms # 
class TransformedSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):  # Constructor method. Accepts dataset object, transformations etc.
        self.subset = subset
        self.transform = transform
    def __getitem__(self, index):  # Enables indexing. applies transformation pipeline and return the pair (x, y)
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
    def __len__(self): 
        return len(self.subset)  # PyTorch uses this length to determine size, iteration, shuffling, batching of the dataset.

# Split #
train_size = int(0.8*len(train_set))
val_size = len(train_set) - train_size
train_subset, val_subset = random_split(train_set, [train_size, val_size])

train_dataset = TransformedSubset(train_subset, transform=transform_train)
val_dataset = TransformedSubset(val_subset, transform=transform_val_test)
test_dataset = TransformedSubset(test_set, transform=transform_val_test)

# Data Loading #
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle = False)

class_names = train_set.classes  # ImageFolder automatically scans the subfolders inside the directory to treat them as classes.
print(f"Classes: {class_names}")

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


### Model Class Definition 
- The model uses 3 separate convolution layers.
- Uses the SiLU activation function rather than ReLU for better performance.
- Uses batch normalization layer for every convolution layer for the utilities. 

In [9]:
class theCNN(nn.Module): 
    def __init__(self, num_classes = 10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size = 3, padding = 1),  # 32 filters over 3 input channels
            nn.BatchNorm2d(32),  # Normalize the output across batch, speeds up convergence, acts as regularizer
            nn.SiLU(),
            # Block 2
            nn.Conv2d(32,64, kernel_size = 3, padding = 1),  # Increase channel capacity to capture more complex, localized combinations of features
            nn.BatchNorm2d(64),  # Normalize the output
            nn.SiLU(),
            nn.MaxPool2d(2, 2),  # Halves the spatial dimensions
            # Block 3
            nn.Conv2d(64, 128, kernel_size = 3, padding = 1),# Extract high-level semantic representations
            nn.BatchNorm2d(128), # Normalize again
            nn.SiLU(),
            nn.MaxPool2d(2, 2)   # Halves the spatial dimensions again
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),   # Output Dimensions; 8*8*128=8192; 1D vector output 
            nn.Linear(128*8*8, 256), # Fully connected layer that combines features and projects into a 256 dimensional space.
            nn.SiLU(),
            nn.Dropout(0.4), # Randomly drop 40% connections to force the network to learn redundancies: prevent overfitting.
            nn.Linear(256, num_classes) # Project the vector down to 10 class score
        )

    def forward(self, x):
        x = self.features(x)  
        x = self.classifier(x)
        return x # Output the result as raw class score after passing through the layers.

model = theCNN().to(DEVICE)

### Loss & Optimization
- Fairly standard epoch layer where the calculations and optimization schemes are declared.

In [10]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()  # Operate in training mode for all layers
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()  # Stop PyTorch from adding the new gradients to the existing ones from previous loops
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()  # Backpropagation
        optimizer.step()  # Gradient descent, update the weights using the calculated gradients by backward()

        running_loss += loss.item()*images.size(0)  # Ensure loss is true weighted avg, even if all batch size is not equal in the last loop
        _,predicted = outputs.max(1)  # outputs.max(1) returns: (max_values, indices) of an image. Keep only index
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item() # .item() converts the single value 'loss' tensor into float
                                                     # .eq(): predicted == labels
    return running_loss/total, 100.0*correct/total

@torch.no_grad()  # Stop PyTorch from building the compute graph- using this function log, reducing memory usage and speeding up calculations.
def evaluate(model, loader, criterion, device):
    model.eval()  # Switch layers to evaluation mode
    running_loss = 0.0
    correct = 0
    total =0 

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item()*images.size(0)  # average loss for the current batch * the batchsize: correctly 
        # weights the loss contributions of the last smaller batch so total epoch loss average remains mathematically valid
        _,predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return running_loss/total, 100.0*correct/total

### Training Loop(Epoch)
- Training loop declaration along with backpropagation methods that will be used in the previous cell.
- **Most importantly:** This cell outputs show all the training epoch results, how the models performance  improved with each epoch.
- The model achieved roughly **77% accuracy for the training data** and **82% accuracy for the validation data**.

In [15]:
# model.parameters()-> iterator over all learnable parameters
# weight_decay -> L2 regularization
optimizer = optim.AdamW(model.parameters(), lr = LEARNING_RATE, weight_decay=1e-4)
# CrossEntropyLoss is the standard for multiclass classification
# reduction -> averages loss over the batch
# label_smoothing -> no hard class(0 or 1) rather soft class(0.9 or something else for the class and rest for others)
criterion = nn.CrossEntropyLoss(reduction = 'mean', label_smoothing = 0.1)
# T_max = full cosine cosine cycle across all epochs
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

best_val_acc = 0.0
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(1, NUM_EPOCHS+1):
    # train_one_epoch() runs forward+backward pass
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    # evaluate() runs model in evaluation mode
    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

    # Adjust Learning rate on validation accuracy after each epoch
    scheduler.step()

    # Logs the values for later analysis
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS}| "
          f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%| "
          f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")

    # Save checkpoint if is the best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'best_val_acc': best_val_acc,
            'hyperparams': {
                'batch_size': BATCH_SIZE,
                'learning_rate': LEARNING_RATE,
                'model': 'theCNN'
            }
        }

        torch.save(checkpoint, CHECKPOINT_DIR/'best_model.pth')
        print(f" ~~New best model saved(val acc: {val_acc:.2f}%)")

Epoch 01/30| Train Loss: 1.8682, Acc: 36.06%| Val Loss: 1.5695, Acc: 50.55%
 ~~New best model saved(val acc: 50.55%)
Epoch 02/30| Train Loss: 1.6761, Acc: 45.85%| Val Loss: 1.4441, Acc: 57.05%
 ~~New best model saved(val acc: 57.05%)
Epoch 03/30| Train Loss: 1.5755, Acc: 51.42%| Val Loss: 1.3389, Acc: 63.07%
 ~~New best model saved(val acc: 63.07%)
Epoch 04/30| Train Loss: 1.5143, Acc: 55.04%| Val Loss: 1.2845, Acc: 66.34%
 ~~New best model saved(val acc: 66.34%)
Epoch 05/30| Train Loss: 1.4628, Acc: 57.95%| Val Loss: 1.2543, Acc: 67.77%
 ~~New best model saved(val acc: 67.77%)
Epoch 06/30| Train Loss: 1.4249, Acc: 59.54%| Val Loss: 1.2385, Acc: 68.98%
 ~~New best model saved(val acc: 68.98%)
Epoch 07/30| Train Loss: 1.3897, Acc: 61.66%| Val Loss: 1.1719, Acc: 71.91%
 ~~New best model saved(val acc: 71.91%)
Epoch 08/30| Train Loss: 1.3625, Acc: 63.08%| Val Loss: 1.1599, Acc: 72.53%
 ~~New best model saved(val acc: 72.53%)
Epoch 09/30| Train Loss: 1.3337, Acc: 64.17%| Val Loss: 1.1475, 

### Test Set Evaluation
- The model performed similarly(**~82%**) for the test data as well. Test data was previously unseen. 

In [20]:
best_checkpoint = torch.load(CHECKPOINT_DIR/'best_model.pth')
model.load_state_dict(best_checkpoint['model_state_dict'])

test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)
print(f"Final Test Accuracy: {test_acc:.2f}%")

Final Test Accuracy: 81.71%


### Test images from outside the CIFAR10 dataset. 
- I downloaded some photos, named them, put them in IMG folder.
- I intentionally chose unusual type or confusing type of photos. Like chose puppies as dogs, chose very old car models, 1 big and 1 small truck etc.
- Test shows great accuracy! Honestly better that what I expected.
- dog-2 was predicted to be a bird! This was the only mistake the model made.
- Looking at the photo, that is a bulldog puppy, with an unusual look, so the model is not that good to predict even that correctly it seems!

In [22]:
from PIL import Image
from pathlib import Path
import torch
import torchvision.transforms as transforms

images_dir = Path("../IMG") 
extensions = ('.png')
image_paths = [p for p in images_dir.iterdir() if p.suffix.lower() in extensions]

inference_transform = transforms.Compose([
    transforms.Resize((32, 32)),  # CIFAR-10 native resolution
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

class_names = train_set.classes # ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

model.eval()
model.to(DEVICE)

for img_path in image_paths:
    try:
        img = Image.open(img_path).convert('RGB')
        
        input_tensor = inference_transform(img).unsqueeze(0).to(DEVICE)
        
        with torch.no_grad():
            outputs = model(input_tensor)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)
            predicted_idx = torch.argmax(probabilities, dim=1).item()
            confidence = probabilities[0][predicted_idx].item() * 100
            
            print(f"File: {img_path.name} | Predicted: {class_names[predicted_idx]} ({confidence:.2f}%)")
            
    except Exception as e:
        print(f"Could not process {img_path.name}: {e}")

File: Bird-1.png | Predicted: bird (87.82%)
File: car-1.png | Predicted: automobile (91.47%)
File: Bird-2.png | Predicted: bird (36.67%)
File: dog-2.png | Predicted: bird (68.98%)
File: Plane-2.png | Predicted: airplane (90.16%)
File: dog-1.png | Predicted: dog (63.20%)
File: car-3.png | Predicted: automobile (91.08%)
File: car-2.png | Predicted: automobile (97.44%)
File: Bird-3.png | Predicted: bird (66.49%)
File: plane-1.png | Predicted: airplane (95.43%)
File: truck-2.png | Predicted: truck (80.78%)
File: truck-1.png | Predicted: truck (69.76%)


### To reuse the model using the git repo
- The model declaration have to be there for the test too.
- Meaning:
  + First have theCNN() model declaration in your test
  + Have all the dependencies needed
  + Then the below code snippet to load the trained parameters into the declared model.
  + You then have this exact model fully reproduced.
  + Check the documentation for more details. 

In [ ]:
import torch
import torchvision.transforms as transforms
from pathlib import Path

DEVICE = torch.device("cpu") # or gpu: cuda, rocm or vulkan
model = theCNN().to(DEVICE)
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2023, 0.1994, 0.2010)
CHECKPOINT_DIR = Path("../Checkpoints")
checkpoint = torch.load(CHECKPOINT_DIR / 'best_model.pth', map_location=DEVICE)

# Restore the saved weights #
model.load_state_dict(checkpoint['model_state_dict'])

model.eval()
print(f"Model loaded!")